# Feature Engineering

In dieser Übung arbeiten wir mit einem [Kaggle Datensatz mit Top 5000 Filmen](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata). Wir wollen den Umsatz eines Films basierend auf seinen sonstigen Features vorhersagen.

## Datensatz

Als erstes müssen wir diesen einlesen (vergiss nicht, den Datensatz vorher zu entpacken). Der Input sind 2 CSV Dateien, die wir beide einlesen wollen.

In [1]:
import pandas as pd

credits = pd.read_csv("archive/tmdb_5000_credits.csv")
movies = pd.read_csv("archive/tmdb_5000_movies.csv")

Wir wollen zuerst die beiden Datensätze mit der [pd.merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) Methode zu einem DataFrame zusammen joinen. Am einfachsten funktioniert das, wenn beide Datensätze einen passenden Index haben. In `tmdb_5000_movies.csv` gibt es eine Spalte `id`, welche mit der Spalte `movie_id` aus `tmdb_5000_credits.csv` übereinstimmt. Wenn wir diese jeweils als index setzen, können wir die beiden DataFrames einfach mergen.

In [2]:
# ADD CODE HERE

credits.set_index("movie_id")
movies.set_index("id")

print(credits.columns)
print(movies.columns)

merged_data = pd.merge(right= movies, left = credits, right_on="id", left_on="movie_id")
merged_data = merged_data.drop(columns=["title_x"])
merged_data = merged_data.rename(columns={"title_y": "title"})
merged_data.drop(columns = ["id"])

print(merged_data.columns)

Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')
Index(['movie_id', 'cast', 'crew', 'budget', 'genres', 'homepage', 'id',
       'keywords', 'original_language', 'original_title', 'overview',
       'popularity', 'production_companies', 'production_countries',
       'release_date', 'revenue', 'runtime', 'spoken_languages', 'status',
       'tagline', 'title', 'vote_average', 'vote_count'],
      dtype='object')


Da wir den Umsatz der Filme vorhersagen wollen, sollten wir nur Einträge behalten, wo auch ein Umsatz (`revenue`) echt größer 0 angegeben ist.

In [3]:
# ADD CODE HERE

#boolean indexing

filtered_df = merged_data[merged_data["revenue"] > 0]

print(len(merged_data))
print(len(filtered_df))

4803
3376


Jetzt sollten wir die Daten aufteilen in einen Teil, den wir vorhersagen wollen (`revenue`) und einen, der alle Spalten mit Außnahme von `revenue` enthält. Indem wir das target von den Features separat halten reduzieren wir das Potential für Fehler.

Mit [DataFrame.drop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html) können wir (mit `axis=1`) Spalten entfernen.

In [4]:
target = filtered_df[["revenue"]]
features = filtered_df.drop(columns=["revenue"])

#print(target.head())
#print(features.head())


## Train-Test Split

Als nächstes wollen wir die Daten unterteilen in Trainings und Test Daten. Mit den Test Daten wollen wir bestimmen, wie gut das Modell generalisiert.

Dafür können wir [sklearn.model_selection.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) verwenden. Wir geben der Methode als Parameter `features` und `target` mit. Die Größe vom Testset konfigurieren wir als `test_size=0.15`. Als Rückgabe sollten wir 4 Datensätze erhalten.

In [5]:
from sklearn.model_selection import train_test_split

# Initialize features_train, features_test, target_train, target_test

# ADD CODE HERE

features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.15)

## Lineare Regression

Der Umsatz ist eine recht große Zahl, PoissonRegression macht daher wenig Sinn. Wir wollen hier den Wert per LinearRegression vorhersagen. Für den Start nehmen wir einfach die numerischen Spalten `["budget", "popularity", "runtime", "vote_average", "vote_count"]` als Features.

Trainiere eine lineare Regression wie gehabt und berechne den RMSE sowohl für das Trainings, als auch das Test Set.

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# ADD CODE HERE
selected_features = ["budget", "popularity", "runtime", "vote_average", "vote_count"]
feature_train = features_train[selected_features]
feature_test = features_test[selected_features]

model1 = LinearRegression()
model1.fit(feature_train, target_train)

#RMSE
#predictions

predictions_train = model1.predict(feature_train)
rmse_train = np.sqrt(mean_squared_error(y_true=target_train, y_pred=predictions_train))
print(f"Train:\n{rmse_train}\n")

predictions_test = model1.predict(feature_test)
rmse_test = np.sqrt(mean_squared_error(y_true=target_test, y_pred=predictions_test))
print(f"Test:\n{rmse_test}\n")


Train:
100460965.19833249

Test:
102066617.25588761



## ColumnTransformer

Wir wollen nun unsere Feature mit [sklearn.preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) auf Mean 0 und Standard Deviation 0 bringen. Da LinearRegression recht robust ist, sollte die Skalierung keinen Unterschied machen, aber das ist ein guter Startpunkt für weitere Transformationen.

Um uns die Arbeit einfacher zu machen wollen wir mit dem [sklearn.compose.ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) angeben, welche Spalten zu bearbeiten sind. Dieser bekommt als Argument eine Liste mit Tripeln

Das Tripel besteht aus
- einen Namen (ein beliebiger String)
- einen Transformer (hier `StandardScaler()`)
- ein oder mehrere Spaltennamen, auf die der Transformer angewendet werden soll (hier `["budget", "popularity", "runtime", "vote_average", "vote_count"]`)

Der Parmeter `remainder` von `ColumnTransformer` steht schon im Default auf `drop`, was angibt, dass Spalten, für die wir keine Angabe gemacht haben, ignoriert werden sollen.

Erstelle einen `ColumnTransformer` und schau, was der aus den Trainings Features macht.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

column_trans = ColumnTransformer(
    [("norm", StandardScaler(), ["budget", "popularity", "runtime", "vote_average", "vote_count"])]
    )
column_trans.fit_transform(features_train)


array([[-0.53615378, -0.4089287 , -0.623704  ,  0.45194144, -0.48757443],
       [ 0.03124958, -0.11888702, -0.19778998, -0.3449977 , -0.35274257],
       [-0.30919243, -0.10933585, -0.623704  ,  0.11039609, -0.16960196],
       ...,
       [-0.71772285, -0.31073453, -0.52905644, -0.5726946 , -0.47964315],
       [-0.24110403,  0.26249782,  0.22812404,  0.67963834, -0.08884704],
       [-0.44536924, -0.10611047,  1.45854233,  1.02118368, -0.29217638]],
      shape=(2869, 5))

## Pipelines

Das nächste sklearn Feature, was wir brauchen ist die [sklearn.pipeline.Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html). Mit dieser können wir verschiedene Transformationen hintereinander laufen lassen (`ColumnTransformer` läuft quasi parallel auf mehreren Spalten eines DataFrames, eine `Pipeline` läuft nacheinander).

Der Parameter von `Pipeline` ist eine Liste von Tupeln.
Jedes Tupel besteht aus
- Einem Namen (ein beliebiger String)
- Einem Transformer

Baue eine Pipeline aus zwei Schritten. Einem "preprocess" Schritt (der ColumnTransformer von gerade). Und einer `LinearRegression` im Anschluss.

Nun können wir `pipeline.fit(features_train, target_train)` aufrufen, um in einem Zug die Paremeter für die `StandardScaler` und die `LinearRegression` zu bestimmen.

Auf der gefitteten Pipeline können wir nun mit `pipeline.predict(features_train)` ebenso in einem Zug Predictions berechnen. Ermittle so den RMSE für das Trainings und das Testset. Diese sollten recht nah an den Ergebnissen von vorhin liegen.

In [8]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([("transform_columns", column_trans),("regression", model1)])

# ADD CODE HERE
pipeline.fit(feature_train, target_train)
predictions = pipeline.predict(feature_train)

Die Nutzung von Pipelines hat viele Vorteile. Wir können so verschiedene Datenverarbeitungsschrittte bündeln und verringern das Risiko für bestimmte Fehler (zum Beispiel in Produktion die Parameter von StandardScaler auf den neuen Daten neu zu bestimmen oder den Scaler ganz zu vergessen). Ausserdem können wir die Pipeline als ganzes serialisieren und zum Beispiel nach Produktion deployen und dort wieder laden:

In [9]:
import pickle
with open("model.pickle", "wb") as f:
    pickle.dump(pipeline, f)
    
with open("model.pickle", "rb") as f:
    loaded_pipeline = pickle.load(f)
    
loaded_pipeline.predict(features_test)

array([[ 5.04708628e+07],
       [ 6.09023135e+07],
       [ 2.74322428e+07],
       [ 2.11872272e+08],
       [ 7.98312456e+07],
       [ 2.60064217e+08],
       [-1.43288405e+07],
       [ 8.50697491e+07],
       [-1.18361802e+06],
       [ 1.42492397e+08],
       [ 1.50121190e+07],
       [ 3.64062128e+07],
       [ 6.23194537e+08],
       [ 8.95974241e+06],
       [ 6.34700351e+07],
       [ 3.47847241e+08],
       [ 6.99553459e+07],
       [ 2.95921066e+08],
       [ 1.35319209e+08],
       [ 2.36049014e+08],
       [ 7.82795141e+07],
       [-5.74259545e+05],
       [ 3.56920567e+07],
       [ 7.66811175e+08],
       [-1.15955588e+07],
       [ 7.83993077e+07],
       [ 1.33731813e+08],
       [ 1.10330850e+08],
       [ 4.70659623e+07],
       [ 2.74733134e+07],
       [ 5.33176336e+07],
       [ 4.87736941e+07],
       [ 1.09848063e+08],
       [ 7.16866420e+06],
       [ 9.21602879e+07],
       [ 2.33132069e+08],
       [ 2.12626418e+07],
       [ 5.94154500e+07],
       [ 6.5

## Bag of words

Die Spalten `title`, `overview` und `original_language` enthalten Text, die wir mit dem bag-of-words Modell modellieren können. Das können wir mit dem [sklearn.feature_extraction.text.CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) erledigen.

Dazu fügen wir weitere Elemente zu unserem `ColumnTransformer` hinzu. Währen der `StandardScaler` für viele Spalten gleichzeitig verwendet werden kann, bekommt jeder `CountVectorizer` immer nur genau eine Spalte angegeben. Begrenze die maximale Anzahl Features mit `max_features=100`.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer

# ADD CODE HERE

column_trans = ColumnTransformer(
    [("norm", StandardScaler(), ["budget", "popularity", "runtime", "vote_average", "vote_count"]),
    ("title", CountVectorizer(max_features=100), "title"),
    ("overview", CountVectorizer(max_features=100), "overview"),
    ("original_language", CountVectorizer(max_features=100), "original_language")]
    )


## Custom Tranformer

Als nächstes wollen wir das Erscheinungsjahr und den Erscheinungsmonat verwenden. Diese müssen wir zunächst aus der Spalte `release_date` extrahieren. Dazu verwenden wir einen Selbstgebauten Transformer, den wir in unserer Pipeline vor dem ColumnTransformer aufrufen. In der `transform` Methdode bekommt unser neuer Transformer den Original Dataframe übergeben und soll diesem zwei neue Spalten `year` und `month` hinzufügen.

Indem wir `pd.to_datetime` auf der `release_date` Spalte aufrufen bekommen wir eine Spalte mit geparsten DatetTime Objekten. Auf dieser kann man dann `.dt.year` oder `.dt.month` aufrufen um eine Spalte mit dem Jahr oder dem Monat zu erhalten. Diese können wir dann dem Input DataFrame hinzufügen.

Um den Monat sinnvoll zu verarbeiten, verwenden wir [sklearn.preprocessing.OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html). Diesen müssen wir dem ColumnTransformer für die neue `month` Spalte hinzufügen.

Denke auch daran, die neue `year` Spalte zum Input für den StandardScaler hinzuzufügen.

In [11]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin


class AddColumns(BaseEstimator, TransformerMixin):        
    def fit(self, X, y=None):
        return self

    def transform(self, X, y = None):
        
        # ADD CODE HERE
        datetime_data = pd.to_datetime(X["release_date"])
        X["year"] = datetime_data.dt.year
        X["month"] = datetime_data.dt.month
        
        return X

# ADD CODE HERE

column_trans = ColumnTransformer(
    [("norm", StandardScaler(), ["budget", "popularity", "runtime", "vote_average", "vote_count", "year"]),
     ("month", OneHotEncoder(), ["month"]),
    ("title", CountVectorizer(max_features=100), "title"),
    ("overview", CountVectorizer(max_features=100), "overview"),
    ("original_language", CountVectorizer(max_features=100), "original_language")]
    )

pipeline = Pipeline([("custom transformer", AddColumns()),("transform_columns", column_trans),("regression", model1)])
pipeline.fit(features_train, target_train)

,steps,"[('custom transformer', ...), ('transform_columns', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('norm', ...), ('month', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Komplexer Input

Die Spalten `genres`, `keywords`, `production_companies`, `production_countries`, `cast` und `crew` sind etwas schwieriger. Hier sind die Informationen im [JSON](https://en.wikipedia.org/wiki/JSON) Format kodiert. Mit `json.loads` können wir einen solchen String in ein Python Objekt konvertieren.

Schreibe eine Methode, welche für einen JSON kodierten Eintrag die einzelnen `"name"` Felder extrahiert und als Komma separierte Liste zurück gibt. Für jeden einzelnen "name" wollen wir Leerzeichen im Eintrag entfernen, damit der ganze Eintrag später vom CountVectorizer als ein Eintrag erkannt wird.

In [12]:
import json

def name_extractor(entry):

    objects = json.loads(entry)
    result = ''

    for obj_index in range(len(objects)):
        
        obj = objects[obj_index]
        result += obj["name"].replace(" ", "")

        if obj_index < len(objects)-1:
            result+= ', '

    return result
    
assert name_extractor('[{"name": "Threshold Entertainment", "id": 4174}, {"name": "Lions Gate Family Entertainment", "id": 6574}]') == 'ThresholdEntertainment, LionsGateFamilyEntertainment'


Erweitere nun die `AddColumns` Klasse, so dass auch für die 6 komplexeren Spalten die Felder extrahiert werden. Achte darauf, dass du den extrahierten Features Spaltennamen gibst, die nicht mit den existierenden kollidieren (um nicht die alten Werte zu überschreiben).
Du kannst auf einer Spalte die Funktion `.apply(name_extractor)` aufrufen, um unseren name_extractor auf alle Elemente anzuwenden und so die transformierte Spalte zu erhalten.

Erweitere dann den `ColumnTransformer`, so dass auf die neuen Spalten auch je ein `CountVectorizer` angewendet wird.

In [13]:
# ADD CODE HERE
class AddColumns(BaseEstimator, TransformerMixin):        
    def fit(self, X, y=None):
        return self

    def transform(self, X, y = None):
        
        # ADD CODE HERE
        datetime_data = pd.to_datetime(X["release_date"])
        X["year"] = datetime_data.dt.year
        X["month"] = datetime_data.dt.month

        #6 komplexe spalten
        spalten = ["genres", "keywords", "production_companies", "production_countries", "cast", "crew"]

        for colname in spalten:
            X[f"{colname}_from_json"] = X[f"{colname}"].apply(name_extractor)
        
        
        return X
    
column_trans = ColumnTransformer([
    ("norm", StandardScaler(),
     ["budget", "popularity", "runtime", "vote_average", "vote_count", "year"]),

    ("month", OneHotEncoder(handle_unknown="ignore"), ["month"]),

    ("title", CountVectorizer(max_features=100), "title"),
    ("overview", CountVectorizer(max_features=100), "overview"),
    ("original_language", CountVectorizer(max_features=100), "original_language"),

    ("genres", CountVectorizer(max_features=100), "genres_from_json"),
    ("keywords", CountVectorizer(max_features=100), "keywords_from_json"),
    ("production_companies", CountVectorizer(max_features=100), "production_companies_from_json"),
    ("production_countries", CountVectorizer(max_features=100), "production_countries_from_json"),
    ("cast", CountVectorizer(max_features=100), "cast_from_json"),
    ("crew", CountVectorizer(max_features=100), "crew_from_json"),
])

pipeline = Pipeline([
    ("custom_transformer", AddColumns()),
    ("transform_columns", column_trans),
    ("regression", LinearRegression())
])
pipeline.fit(features_train, target_train)

,steps,"[('custom_transformer', ...), ('transform_columns', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('norm', ...), ('month', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Aus dem `ColumnTransformer` kann man mit `get_feature_names_out` die Namen der ausgegebenen Features bekommen.
Zu jedem von diesem können wir die Koeffizienten der Linearen Regression erhalten. Drucke zu jedem Feature den zugehörigen Koeffizienten.

In [14]:
# ADD CODE HERE
feature_names = pipeline.named_steps[
    "transform_columns"
].get_feature_names_out()

coefficients = pipeline.named_steps[
    "regression"
].coef_

for name, coef in zip(feature_names, coefficients):
    print(name, coef)

norm__budget [ 6.56229718e+07  1.41039893e+07  5.55548411e+06 -4.82239346e+06
  9.90081248e+07 -4.29873902e+06 -1.01460960e+07  2.33214776e+06
 -2.30130160e+06 -1.33451235e+05  9.08881755e+06  8.44509295e+06
 -5.01376863e+06 -5.94523255e+06  2.00255359e+06 -1.27385615e+07
  3.26795669e+06  1.11418429e+07  2.33680408e+07 -6.22400031e+06
 -4.22403424e+06 -1.92261143e+06 -9.58661617e+06  6.27186253e+06
  1.58756084e+07 -8.19811656e+06 -2.16862730e+07 -1.14815066e+07
  3.47831834e+07 -2.15023386e+07  8.75259808e+06 -1.56846559e+07
  5.09790434e+05 -3.11656268e+06 -2.61122637e+07  4.21261178e+07
  9.40018162e+07  2.36463788e+07 -2.34488078e+07 -2.71534507e+07
  2.89887024e+07 -5.81816160e+07  5.17368370e+06  2.73295185e+07
  1.52919025e+07  2.06264201e+07  3.19080249e+07 -1.81318033e+07
  2.94625248e+07 -8.19243491e+06 -1.49289441e+07  2.68751691e+07
 -3.42748114e+07  2.00018172e+07 -6.79873407e+07  9.43589524e+07
 -1.23989127e+07  3.96755332e+07  1.31585842e+07 -2.33766184e+07
  1.32678004

In [15]:
#RMSE
predictions_train = pipeline.predict(features_train)
rmse_train = np.sqrt(mean_squared_error(y_true=target_train, y_pred=predictions_train))
print(f"Train:\n{rmse_train}\n")

predictions_test = pipeline.predict(features_test)
rmse_test = np.sqrt(mean_squared_error(y_true=target_test, y_pred=predictions_test))
print(f"Test:\n{rmse_test}\n")

Train:
77097296.26636931

Test:
105327364.52023982



## Overfitting

Voraussichtlich ist inzwischen der Fehler auf dem Trainingsset deutlich geringer, als auf dem Testset. Hierbei spricht man von Overfitting. Wir haben ein Modell, was schlecht generalisiert, weil es sich zu genau an die Trainingsdaten anpassen kann.

Das können wir jetzt auf die Spitze treiben, indem wir in der Pipeline zwischen `ColumnTransformer` und `LinearRegression` noch 
[sklearn.preprocessing.PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) vom Grad 2 einfügen.

In [16]:
from sklearn.preprocessing import PolynomialFeatures

pipeline = Pipeline([
    ("custom_transformer", AddColumns()),
    ("transform_columns", column_trans),
    ("pol_features", PolynomialFeatures(degree=2)),
    ("regression", LinearRegression())
])
pipeline.fit(features_train, target_train)

,steps,"[('custom_transformer', ...), ('transform_columns', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('norm', ...), ('month', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
#RMSE
predictions_train = pipeline.predict(features_train)
rmse_train = np.sqrt(mean_squared_error(y_true=target_train, y_pred=predictions_train))
print(f"Train:\n{rmse_train}\n")

predictions_test = pipeline.predict(features_test)
rmse_test = np.sqrt(mean_squared_error(y_true=target_test, y_pred=predictions_test))
print(f"Test:\n{rmse_test}\n")

Train:
45738.36915242643

Test:
101417056.25745386



Ja, aus RMSE geht hervor, dass Overfitting extrem ist.
Das ist passiert, weil PolynomialFeatures eine große Menge an zusätzlichen Features erzeugt indem die Interactions zwischen den Features eingeführt werden.
Dadurch generalisiert das Modell die Daten nicht, sondern erzeugt einzelne Regeln für viele Trainingsbeispiele und "merkt" sich dadurch die Trainingsdaten.